In [9]:
# --- BLOCK 1: SETUP ---
import pandas as pd
import numpy as np
import sys
import os
import joblib

# Time-Series & Machine Learning
from prophet import Prophet
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor

# Visualization
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

# Local Pipeline Config
sys.path.append('..') 

# Safety check: Ensure our output folders exist before we save anything!
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../models', exist_ok=True)

print("✅ Stage 1 Complete: Environment synchronized and directories ready.")

✅ Stage 1 Complete: Environment synchronized and directories ready.


In [10]:
# --- BLOCK 2: THE REAL SCIENCE DATA INGESTION (One Earth Fix) ---
import requests
from io import StringIO
import pandas as pd

print("📡 Accessing Real Global Climate Data...")

# 1. Load local real CO2 and GHG data
co2_df = pd.read_csv('../data/raw/owid-co2-data.csv') 
ghg_df = pd.read_csv('../data/raw/total-ghg-emissions.csv') 

# 🚀 RENAME COLUMNS IMMEDIATELY so our filters don't crash!
ghg_df = ghg_df.rename(columns={
    'Entity': 'country',
    'Code': 'iso_code',
    'Year': 'year',
    'Annual greenhouse gas emissions including land use': 'Total_GHG'
})

# ---------------------------------------------------------
# 🚀 THE "ONE EARTH" FIX 
# We drop nulls and OWID aggregates immediately so they don't 
# cross-multiply and poison our merge.
# ---------------------------------------------------------
co2_df = co2_df.dropna(subset=['iso_code'])
co2_df = co2_df[~co2_df['iso_code'].str.startswith('OWID_')]

ghg_df = ghg_df.dropna(subset=['iso_code'])
ghg_df = ghg_df[~ghg_df['iso_code'].str.startswith('OWID_')]


# 2. Fetch REAL TEMPERATURE DATA (Bypassing OWID's Bot Protection)
temp_url = "https://ourworldindata.org/grapher/annual-temperature-anomalies.csv?v=1&csvType=full&useColumnShortNames=false"
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
}
response = requests.get(temp_url, headers=headers)

if response.status_code == 200:
    temp_df = pd.read_csv(StringIO(response.text))
else:
    raise Exception(f"❌ Server blocked the request! Status Code: {response.status_code}")

temp_df = temp_df.rename(columns={
    'Entity': 'country', 
    'Code': 'iso_code', 
    'Year': 'year', 
    'Temperature anomaly': 'Temp_Anomaly'
})

# 4. Merge the Science Data Together
# 🌟 ADDED 'methane' AND 'nitrous_oxide' TO THE MERGE!
df = pd.merge(co2_df[['country', 'iso_code', 'year', 'population', 'co2', 'methane', 'nitrous_oxide']], 
              ghg_df[['iso_code', 'year', 'Total_GHG']], 
              on=['iso_code', 'year'], how='left')

df = pd.merge(df, temp_df[['iso_code', 'year', 'Temp_Anomaly']], 
              on=['iso_code', 'year'], how='inner')

# 5. Rename for our Web App standard
df = df.rename(columns={
    'country': 'Real_Country_Name',
    'year': 'Year',
    'population': 'Population',
    'co2': 'CO2_Emissions',
    'methane': 'Methane_Emissions',             
    'nitrous_oxide': 'Nitrous_Oxide_Emissions'  
})

# 6. Cleanup & Baselines
df['Total_GHG'] = df['Total_GHG'].fillna(0)
df['CO2_Emissions'] = df['CO2_Emissions'].fillna(0)
df['Population'] = df['Population'].fillna(0)
df['Methane_Emissions'] = df['Methane_Emissions'].fillna(0)             
df['Nitrous_Oxide_Emissions'] = df['Nitrous_Oxide_Emissions'].fillna(0) 

df['Average_Temperature'] = 14.0 + df['Temp_Anomaly'] 

print(f"✅ Stage 2 Complete: Clean 'One Earth' Data Loaded! {len(df)} records merged.")

📡 Accessing Real Global Climate Data...
✅ Stage 2 Complete: Clean 'One Earth' Data Loaded! 15725 records merged.


In [11]:
# --- BLOCK 3: ENGINEERING (The Arrhenius Physics Upgrade) ---
import numpy as np 

print("⚙️ Engineering Physics Features...")

# 1. MOVING AVERAGES
df['Temp_Moving_Avg'] = df.groupby('Real_Country_Name')['Temp_Anomaly'].transform(
    lambda x: x.rolling(window=10, min_periods=1).mean()
)

# 2. TIME DIMENSIONS
df['Decade'] = (df['Year'] // 10) * 10

# ---------------------------------------------------------
# 🌟 THE ARRHENIUS UPGRADE: LOGARITHMIC CARBON 🌟
# ---------------------------------------------------------
df = df.sort_values(by=['Real_Country_Name', 'Year'])
df['Cumulative_CO2'] = df.groupby('Real_Country_Name')['CO2_Emissions'].cumsum()

# We apply the natural log to mimic the physics of the atmosphere!
df['Log_Cumulative_CO2'] = np.log1p(df['Cumulative_CO2']) 

# 3. EXPORT THE PURE MASTER FILE
df.to_csv('../data/processed/supreme_dataset.csv', index=False)

print("✅ Stage 3 Complete: Arrhenius Physics injected and saved!")

⚙️ Engineering Physics Features...
✅ Stage 3 Complete: Arrhenius Physics injected and saved!


In [12]:
# --- BLOCK 4: THE ULTIMATE AI WORKSHOP (Bayesian Physics Upgrade) ---
import joblib
import pandas as pd
import numpy as np
from sklearn.linear_model import BayesianRidge # 🚀 THE NEW ENGINE
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor

print("🧠 Initializing the Bayesian Physics Workshop...")

# 1. CREATE GLOBAL DATA
global_df = df.groupby('Year').agg({
    'CO2_Emissions': 'sum',
    'Methane_Emissions': 'sum',           
    'Nitrous_Oxide_Emissions': 'sum',     
    'Population': 'sum',
    'Temp_Anomaly': 'mean', 
    'Cumulative_CO2': 'sum'
}).reset_index()

# 🚀 APPLY ARRHENIUS LOGARITHM
global_df['Log_Cumulative_CO2'] = np.log1p(global_df['Cumulative_CO2'])

# 🌊 OCEAN INERTIA (10-YEAR LAG)
global_df['Target_Temp_Anomaly'] = global_df['Temp_Anomaly'].shift(-10)
train_df = global_df.dropna(subset=['Target_Temp_Anomaly'])

# 2. PREPARE THE TRAINING DATA
features = ['Year', 'CO2_Emissions', 'Log_Cumulative_CO2', 'Population', 'Methane_Emissions', 'Nitrous_Oxide_Emissions']
X = train_df[features]
y = train_df['Target_Temp_Anomaly']

# 3. SCALE THE BRAIN
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 4. TRAIN THE BAYESIAN PHYSICS ENGINE
print("⚡ Training the BayesianRidge Engine...")
ai_model = BayesianRidge() 
ai_model.fit(X_scaled, y)

# 5. SAVE THE RECALIBRATED MODELS
joblib.dump(ai_model, '../models/supreme_nn_model.pkl') 
joblib.dump(scaler, '../models/supreme_scaler.pkl')
print("✅ Global AI trained and saved.")

# Note: We deleted Prophet! The Bayesian model handles it all now.

# 6. FEATURE IMPORTANCE
rf_explainer = RandomForestRegressor(random_state=42)
rf_explainer.fit(X, y)
importance_df = pd.DataFrame({
    "Feature": features,
    "Importance": rf_explainer.feature_importances_
})
importance_df.to_csv("../data/processed/feature_importance.csv", index=False)
print("🎉 PIPELINE COMPLETE! We are officially using Bayesian Probabilities.")

🧠 Initializing the Bayesian Physics Workshop...
⚡ Training the BayesianRidge Engine...
✅ Global AI trained and saved.
🎉 PIPELINE COMPLETE! We are officially using Bayesian Probabilities.
